In [1]:
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import os
import csv
import torch
from transformers import DataCollatorForLanguageModeling
from transformers import AutoTokenizer
from transformers import GPT2Config, GPT2LMHeadModel
from transformers import TrainingArguments, Trainer
from transformers import TrainerCallback, TrainerControl, TrainerState

In [2]:
HF_TOKEN = userdata.get('HF_TOKEN')
login(HF_TOKEN)

In [3]:
dataset = load_dataset("azizdevlab/uzbek_corpus")

README.md:   0%|          | 0.00/367 [00:00<?, ?B/s]

data/train-00000-of-00006.parquet:   0%|          | 0.00/72.3M [00:00<?, ?B/s]

data/train-00001-of-00006.parquet:   0%|          | 0.00/72.2M [00:00<?, ?B/s]

data/train-00002-of-00006.parquet:   0%|          | 0.00/73.7M [00:00<?, ?B/s]

data/train-00003-of-00006.parquet:   0%|          | 0.00/73.2M [00:00<?, ?B/s]

data/train-00004-of-00006.parquet:   0%|          | 0.00/73.2M [00:00<?, ?B/s]

data/train-00005-of-00006.parquet:   0%|          | 0.00/67.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/692471 [00:00<?, ? examples/s]

In [4]:
tokenizer = AutoTokenizer.from_pretrained("azizdevlab/gpt2-small-uzbek")

tokenizer.json: 0.00B [00:00, ?B/s]

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 692471
    })
})

In [6]:
dataset = dataset['train'].train_test_split(test_size=0.05)

In [7]:
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 657847
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 34624
    })
})

In [8]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [9]:
config = GPT2Config(
    vocab_size=tokenizer.vocab_size, #20000
    n_positions=768,
    n_ctx=768,
    n_embd=704,
    n_layer=12,
    n_head=11,
    n_inner=2816,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)
model =GPT2LMHeadModel(config)

In [10]:
log_file = "./loss_log.csv"

# Если файла нет — создаём и пишем заголовок
if not os.path.exists(log_file):
    with open(log_file, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["step", "train_loss", "eval_loss"])

In [11]:
class LogLossCallback(TrainerCallback):
    def __init__(self, log_file):
        self.log_file = log_file
        self.last_train_loss = None
        self.last_eval_loss = None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return

        if "loss" in logs:
            self.last_train_loss = logs["loss"]

        if "eval_loss" in logs:
            self.last_eval_loss = logs["eval_loss"]

        with open(self.log_file, "a", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                state.global_step,
                self.last_train_loss,
                self.last_eval_loss
            ])

In [23]:
training_args = TrainingArguments(
    output_dir="./model_out",
    # training
    per_device_train_batch_size=16,#32
    per_device_eval_batch_size=32, #32
    learning_rate=2.5e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2164,

    # Evaluation / Logging
    eval_strategy="steps",
    eval_steps=500,
    logging_strategy="steps",
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    save_strategy="epoch",
    num_train_epochs=2,
    report_to="none",  # важно, чтобы не было конфликтов
)

In [13]:
# training_args = TrainingArguments(
#     output_dir="gpt2_colab_kaggle",

#     # Основное
#     num_train_epochs=1,
#     per_device_train_batch_size=4,
#     per_device_eval_batch_size=4,
#     gradient_accumulation_steps=4,
#     learning_rate=2.5e-4,
#     lr_scheduler_type="cosine",
#     warmup_steps=2164,

#     weight_decay=0.01,

#     # Evaluation / Logging
#     evaluation_strategy="steps",
#     eval_steps=500,
#     logging_strategy="steps",
#     logging_steps=500,
#     save_strategy="steps",
#     save_steps=5000,
#     save_total_limit=10,

#     # Mixed precision / memory optimizations
#     fp16=True,
#     gradient_checkpointing=True,

#     # Reproducibility / logging
#     seed=42,
#     report_to="wandb",
# )

In [24]:
trainer = Trainer(model=model,
                 args = training_args,
                 train_dataset=dataset["train"],
                 eval_dataset=dataset["test"],
                 data_collator = data_collator,
                 callbacks=[LogLossCallback(log_file)]
                )

In [15]:
# pip -q install -U --force-reinstall "accelerate>=0.27.0"

In [16]:
import transformers, accelerate
print(transformers.__version__)
print(accelerate.__version__)

5.0.0
1.12.0


In [25]:
tokenizer.pad_token = tokenizer.eos_token
model.config.loss_type = "ForCausalLMLoss"

In [ ]:
trainer.train()

Step,Training Loss,Validation Loss


In [20]:
tokenizer.special_tokens_map

{'bos_token': '<|endoftext|>',
 'eos_token': '<|endoftext|>',
 'unk_token': '<|endoftext|>'}